# Three-way email classification comparison

This notebook converts the existing CLM email-classification evaluation into an apples-to-apples comparison of:

1. **CLM** through the local or remote System One API;
2. **Laya** through its local Python `Router`;
3. **Microsoft-Decision-1** through a Microsoft Foundry deployment.

Every backend receives the same email state, six labels, option descriptions, and classification instruction. The notebook evaluates the published 2,697-row test split from `jason23322/high-accuracy-email-classifier` and reports coverage, accuracy, macro/weighted F1, log loss, Brier score, top-2 accuracy, calibration, latency, per-class behavior, and paired head-to-head comparisons.

> The test split is for final comparison, not prompt or model selection. Develop criteria and fine-tuning choices on a training-derived holdout. Microsoft-Decision-1 can remain disabled until a Foundry deployment is available.

## Evaluation contract

- Preserve the source notebook's six-class taxonomy and balanced test split.
- Send one typed `choice` question per email.
- Measure client-observed end-to-end latency for every backend.
- Use the maximum class probability as the comparable confidence measure. CLM's margin-based `confidence` is retained separately but is not compared directly with other backends.
- Save one resumable prediction file per backend.
- Reject stale checkpoints when the dataset, labels, criteria, instruction, model, or backend configuration changes.
- Stop after repeated failures so an authentication or service outage does not produce thousands of invalid rows.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import statistics
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split

CLIENT_ROOT = Path(r"H:\masterswork\gitrash\CLM\CLMClient")
ENV_PATH = CLIENT_ROOT / ".env"
ARTIFACT_DIR = CLIENT_ROOT / "notebooks" / "artifacts" / "email_three_way"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "jason23322/high-accuracy-email-classifier"
TEST_FILE = "test.json"
LABELS = ["forum", "promotions", "social_media", "spam", "updates", "verify_code"]
RANDOM_STATE = 42
EVALUATION_SAMPLE_SIZE = None  # None evaluates all 2,697 test rows.
CHECKPOINT_EVERY = 25
MAX_CONSECUTIVE_FAILURES = 3
REQUEST_TIMEOUT_SECONDS = 120.0
BOOTSTRAP_REPLICATES = 2_000

# Safe defaults: initialize and smoke-test local backends, but do not start 2,697-row runs accidentally.
ENABLE_BACKENDS = {"clm": True, "laya": True, "microsoft_decision_1": False}
RUN_SMOKE_TESTS = True
RUN_EVALUATIONS = {"clm": False, "laya": False, "microsoft_decision_1": False}

pd.set_option("display.max_colwidth", 120)
print(f"Artifacts: {ARTIFACT_DIR}")

## Secure configuration

The notebook loads `.env` without printing secrets.

| Backend | Configuration |
|---|---|
| Dataset | `HUGGINGFACE_API_KEY` for the gated Hugging Face dataset |
| CLM | `CLM_BASE_URL`, optional `CLM_API_KEY`, and `CLM_MODEL` (default `clm-latest`) |
| Laya | Optional `LAYA_DEVICE` (`cpu`, `cuda`, or platform-supported value) |
| Microsoft-Decision-1 | `AZURE_ENDPOINT`, `AZURE_DECISION_DEPLOYMENT`, and either `AZURE_API_KEY` or `AZURE_ENTRA_TOKEN` |

For a long Microsoft run, use a token-refreshing credential rather than a manually copied Entra token. The simple token option here is intended for controlled notebook evaluation.

In [ ]:
if ENV_PATH.is_file():
    load_dotenv(ENV_PATH)
else:
    print(f"No .env file at {ENV_PATH}; using process environment variables.")

HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY") or os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HUGGINGFACE_API_KEY or HF_TOKEN to load the gated email dataset.")

CLM_BASE_URL = os.getenv("CLM_BASE_URL", "http://127.0.0.1:8700").rstrip("/")
CLM_MODEL = os.getenv("CLM_MODEL", "clm-latest")
LAYA_DEVICE = os.getenv("LAYA_DEVICE")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "").rstrip("/")
AZURE_DECISION_DEPLOYMENT = os.getenv("AZURE_DECISION_DEPLOYMENT", "")

print({
    "dataset_token_configured": bool(HF_TOKEN),
    "clm_base_url": CLM_BASE_URL,
    "clm_model": CLM_MODEL,
    "laya_device": LAYA_DEVICE or "automatic",
    "azure_configured": bool(AZURE_ENDPOINT and AZURE_DECISION_DEPLOYMENT),
    "enabled_backends": [name for name, enabled in ENABLE_BACKENDS.items() if enabled],
})

## Load and validate the held-out test data

Schema, labels, missing values, and duplicate IDs are checked explicitly. A stratified subset can be used for pipeline development, but final comparisons should use the complete test split.

In [ ]:
dataset_url = f"hf://datasets/{DATASET_ID}/{TEST_FILE}"
test_df = pd.read_json(dataset_url, storage_options={"token": HF_TOKEN})

expected_columns = {"id", "subject", "body", "text", "category", "category_id"}
missing_columns = expected_columns.difference(test_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")
observed_labels = set(test_df["category"].dropna().unique())
if observed_labels != set(LABELS):
    raise ValueError(f"Unexpected labels: {sorted(observed_labels)}")
if test_df["id"].duplicated().any():
    raise ValueError("Test split contains duplicate IDs.")
if test_df[["subject", "body", "category"]].isna().any().any():
    raise ValueError("Test split contains missing classification fields.")

if EVALUATION_SAMPLE_SIZE is None:
    evaluation_df = test_df.copy()
elif not 0 < EVALUATION_SAMPLE_SIZE <= len(test_df):
    raise ValueError("EVALUATION_SAMPLE_SIZE must be None or between 1 and the test-set size.")
else:
    evaluation_df, _ = train_test_split(
        test_df,
        train_size=EVALUATION_SAMPLE_SIZE,
        stratify=test_df["category"],
        random_state=RANDOM_STATE,
    )

evaluation_df = evaluation_df.sort_values("id").reset_index(drop=True)
print(f"Test rows: {len(test_df):,}; evaluation rows: {len(evaluation_df):,}")
display(evaluation_df["category"].value_counts().reindex(LABELS).rename("count").to_frame())

## One shared typed decision

The criteria below are copied from the source CLM notebook. They are descriptions rather than test examples, avoiding direct label leakage. All three systems receive the same plain dictionary rather than backend-specific question classes.

In [ ]:
CATEGORY_CRITERIA = {
    "forum": "Community forum conversations, discussion threads, group posts, replies, or member activity notifications.",
    "promotions": "Legitimate marketing, sales, discounts, product offers, newsletters, or commercial campaigns.",
    "social_media": "Notifications from social networks about follows, likes, mentions, direct activity, groups, or recommended content.",
    "spam": "Unsolicited, deceptive, suspicious, irrelevant, or low-quality bulk messages, including scams and phishing-like content.",
    "updates": "Operational, account, product, service, order, billing, travel, or informational status updates that are not verification codes.",
    "verify_code": "One-time passwords, authentication codes, account verification codes, login codes, or security confirmation tokens.",
}
CATEGORY_INSTRUCTION = (
    "Classify this email into exactly one category. Use the subject and body together. "
    "Choose the most specific category represented by the email's primary purpose."
)
CATEGORY_QUESTION = {
    "type": "choice",
    "instructions": CATEGORY_INSTRUCTION,
    "criteria": CATEGORY_CRITERIA,
}

def email_state(row: pd.Series) -> dict[str, str]:
    return {"subject": str(row["subject"]).strip(), "body": str(row["body"]).strip()}

display(pd.Series(email_state(evaluation_df.iloc[0]), name="shared state").to_frame())

## Backend adapters

Each adapter returns the same normalized response. Backend-native output remains available in the row-level probability columns. CLM uses temperature 1.0 so probabilities are not artificially sharpened. Laya and Microsoft-Decision-1 use their service defaults because their public quick-start contracts do not expose an equivalent portable temperature setting.

In [ ]:
@dataclass
class NormalizedPrediction:
    model: str
    choice: str
    probabilities: dict[str, float]
    native_confidence: float | None
    input_tokens: int | None
    reported_latency_ms: float | None


@dataclass
class Backend:
    name: str
    model_id: str
    predict: Callable[[dict[str, str]], NormalizedPrediction]


def validate_choice(choice: Any, probabilities: Any) -> tuple[str, dict[str, float]]:
    if not isinstance(choice, str) or choice not in LABELS:
        raise ValueError(f"Backend returned unsupported choice: {choice!r}")
    if not isinstance(probabilities, dict):
        raise TypeError("Backend returned no probability mapping.")
    normalized = {label: float(probabilities.get(label, 0.0)) for label in LABELS}
    if any(not np.isfinite(value) or value < 0 for value in normalized.values()):
        raise ValueError(f"Backend returned invalid probabilities: {normalized}")
    total = sum(normalized.values())
    if not np.isclose(total, 1.0, atol=1e-3):
        raise ValueError(f"Probabilities sum to {total}, not 1.0: {normalized}")
    normalized = {label: value / total for label, value in normalized.items()}
    return choice, normalized


def build_clm_backend() -> Backend:
    from clm import CLMClient

    api_key = os.getenv("CLM_API_KEY")
    health_response = requests.get(f"{CLM_BASE_URL}/health", timeout=10)
    health_response.raise_for_status()
    health = health_response.json()
    if not health.get("ok") or health.get("embedder") is not True:
        raise RuntimeError(f"Real CLM service is not healthy: {health}")
    if health.get("mock") is True:
        raise RuntimeError("Mock CLM server detected; mock output cannot measure model quality.")

    client = CLMClient(base_url=CLM_BASE_URL, api_key=api_key, model=CLM_MODEL)
    available = [item["name"] for item in client.models()]
    if CLM_MODEL not in available:
        raise RuntimeError(f"Configured CLM model {CLM_MODEL!r} is unavailable: {available}")

    def predict(state: dict[str, str]) -> NormalizedPrediction:
        response = client.system_one(
            state=state,
            questions={"category": CATEGORY_QUESTION},
            temperature=1.0,
        )
        answer = response.answers["category"]
        choice, probabilities = validate_choice(answer.choice, answer.probabilities)
        return NormalizedPrediction(
            model=response.model,
            choice=choice,
            probabilities=probabilities,
            native_confidence=float(answer.confidence),
            input_tokens=response.usage.input_tokens,
            reported_latency_ms=response.latency_ms,
        )

    return Backend("clm", CLM_MODEL, predict)


def build_laya_backend() -> Backend:
    try:
        from laya import Router
    except ImportError as exc:
        raise RuntimeError("Install Laya with: python -m pip install laya") from exc

    kwargs: dict[str, Any] = {"preload": False}
    if LAYA_DEVICE:
        kwargs["device"] = LAYA_DEVICE
    router = Router(**kwargs)

    def predict(state: dict[str, str]) -> NormalizedPrediction:
        payload = router.predict(state, {"category": CATEGORY_QUESTION})
        if not isinstance(payload, dict) or not isinstance(payload.get("answers"), dict):
            raise ValueError(f"Laya returned an invalid response: {payload!r}")
        answer = payload["answers"].get("category")
        if not isinstance(answer, dict):
            raise ValueError(f"Laya returned no category answer: {payload!r}")
        choice, probabilities = validate_choice(answer.get("choice"), answer.get("probabilities"))
        routing = payload.get("routing") if isinstance(payload.get("routing"), dict) else {}
        usage = payload.get("usage") if isinstance(payload.get("usage"), dict) else {}
        return NormalizedPrediction(
            model=str(routing.get("model") or payload.get("model") or "laya"),
            choice=choice,
            probabilities=probabilities,
            native_confidence=float(answer["confidence"]) if answer.get("confidence") is not None else None,
            input_tokens=usage.get("input_tokens"),
            reported_latency_ms=None,
        )

    return Backend("laya", "router", predict)


def build_microsoft_backend() -> Backend:
    if not AZURE_ENDPOINT or not AZURE_DECISION_DEPLOYMENT:
        raise RuntimeError("Set AZURE_ENDPOINT and AZURE_DECISION_DEPLOYMENT.")
    if api_key := os.getenv("AZURE_API_KEY"):
        headers = {"api-key": api_key}
    elif entra_token := os.getenv("AZURE_ENTRA_TOKEN"):
        headers = {"Authorization": "Bearer " + entra_token}
    else:
        raise RuntimeError("Set AZURE_API_KEY or AZURE_ENTRA_TOKEN.")
    endpoint = f"{AZURE_ENDPOINT}/providers/microsoft/v1/systemone"

    def predict(state: dict[str, str]) -> NormalizedPrediction:
        response = requests.post(
            endpoint,
            headers={"Content-Type": "application/json", **headers},
            json={
                "model": AZURE_DECISION_DEPLOYMENT,
                "state": state,
                "questions": {"category": CATEGORY_QUESTION},
            },
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        if not response.ok:
            raise RuntimeError(f"Microsoft-Decision-1 returned HTTP {response.status_code}: {response.text[:1000]}")
        payload = response.json()
        if not isinstance(payload, dict) or not isinstance(payload.get("answers"), dict):
            raise ValueError(f"Microsoft-Decision-1 returned an invalid response: {payload!r}")
        answer = payload["answers"].get("category")
        if not isinstance(answer, dict):
            raise ValueError(f"Microsoft-Decision-1 returned no category answer: {payload!r}")
        choice, probabilities = validate_choice(answer.get("choice"), answer.get("probabilities"))
        usage = payload.get("usage") if isinstance(payload.get("usage"), dict) else {}
        return NormalizedPrediction(
            model=str(payload.get("model") or "microsoft-decision-1"),
            choice=choice,
            probabilities=probabilities,
            native_confidence=float(answer["confidence"]) if answer.get("confidence") is not None else None,
            input_tokens=usage.get("input_tokens"),
            reported_latency_ms=None,
        )

    return Backend("microsoft_decision_1", AZURE_DECISION_DEPLOYMENT, predict)

## Initialize enabled backends

Initialization failures are shown explicitly. They do not prevent analysis of completed checkpoints from another backend.

In [ ]:
BACKEND_BUILDERS = {
    "clm": build_clm_backend,
    "laya": build_laya_backend,
    "microsoft_decision_1": build_microsoft_backend,
}
backends: dict[str, Backend] = {}
initialization_errors: list[dict[str, str]] = []

for backend_name, is_enabled in ENABLE_BACKENDS.items():
    if not is_enabled:
        continue
    try:
        backends[backend_name] = BACKEND_BUILDERS[backend_name]()
    except (ImportError, OSError, RuntimeError, ValueError, requests.RequestException) as exc:
        initialization_errors.append({"backend": backend_name, "error": f"{type(exc).__name__}: {exc}"})

if initialization_errors:
    display(pd.DataFrame(initialization_errors))
print("Ready backends:", sorted(backends))

## Single-example smoke tests

This verifies response shape and probability normalization before any full evaluation. Email text is not printed.

In [ ]:
smoke_rows: list[dict[str, Any]] = []
if not RUN_SMOKE_TESTS:
    print("Smoke tests disabled.")
else:
    smoke_row = evaluation_df.iloc[0]
    for backend_name, backend in backends.items():
        started = time.perf_counter()
        try:
            prediction = backend.predict(email_state(smoke_row))
            smoke_rows.append({
                "backend": backend_name,
                "expected": smoke_row["category"],
                "predicted": prediction.choice,
                "max_probability": max(prediction.probabilities.values()),
                "latency_ms": (time.perf_counter() - started) * 1000,
                "status": "ok",
            })
        except (OSError, RuntimeError, ValueError, KeyError, TypeError, requests.RequestException) as exc:
            smoke_rows.append({
                "backend": backend_name,
                "expected": smoke_row["category"],
                "predicted": None,
                "max_probability": np.nan,
                "latency_ms": (time.perf_counter() - started) * 1000,
                "status": f"{type(exc).__name__}: {exc}",
            })
    display(pd.DataFrame(smoke_rows))

## Resumable inference

Set a backend's `RUN_EVALUATIONS` value to `True` after its smoke test passes. Each backend writes an independent checkpoint under `artifacts/email_three_way/`. Configuration fingerprints prevent accidental reuse after a material experiment change.

In [ ]:
probability_columns = [f"p_{label}" for label in LABELS]
result_columns = [
    "id", "expected_category", "predicted_category", "native_confidence", "max_probability",
    "latency_ms", "reported_latency_ms", "model", "input_tokens", "error", *probability_columns,
]


def experiment_config(backend: Backend) -> dict[str, Any]:
    return {
        "dataset": DATASET_ID,
        "dataset_file": TEST_FILE,
        "evaluation_rows": len(evaluation_df),
        "evaluation_ids_sha256": hashlib.sha256("\n".join(evaluation_df["id"].astype(str)).encode()).hexdigest(),
        "labels": LABELS,
        "criteria": CATEGORY_CRITERIA,
        "instruction": CATEGORY_INSTRUCTION,
        "backend": backend.name,
        "model_id": backend.model_id,
        "clm_temperature": 1.0 if backend.name == "clm" else None,
    }


def config_fingerprint(config: dict[str, Any]) -> str:
    payload = json.dumps(config, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()


def checkpoint_paths(backend_name: str) -> tuple[Path, Path]:
    return (
        ARTIFACT_DIR / f"email_predictions_{backend_name}.csv",
        ARTIFACT_DIR / f"email_predictions_{backend_name}.metadata.json",
    )


def load_checkpoint(backend: Backend) -> pd.DataFrame:
    prediction_path, metadata_path = checkpoint_paths(backend.name)
    config = experiment_config(backend)
    fingerprint = config_fingerprint(config)
    if not prediction_path.exists() and not metadata_path.exists():
        return pd.DataFrame(columns=result_columns)
    if not prediction_path.exists() or not metadata_path.exists():
        raise RuntimeError(f"Incomplete checkpoint for {backend.name}; remove both files and rerun.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("fingerprint") != fingerprint:
        raise RuntimeError(
            f"Stale checkpoint for {backend.name}. Move or remove {prediction_path.name} and its metadata before rerunning."
        )
    frame = pd.read_csv(prediction_path)
    if frame["id"].duplicated().any():
        raise ValueError(f"Checkpoint for {backend.name} contains duplicate IDs.")
    unexpected = set(frame["id"].astype(str)).difference(evaluation_df["id"].astype(str))
    if unexpected:
        raise ValueError(f"Checkpoint for {backend.name} contains IDs outside the evaluation set.")
    return frame


def save_checkpoint(backend: Backend, frame: pd.DataFrame) -> None:
    prediction_path, metadata_path = checkpoint_paths(backend.name)
    config = experiment_config(backend)
    frame.sort_values("id").to_csv(prediction_path, index=False)
    metadata_path.write_text(json.dumps({
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "fingerprint": config_fingerprint(config),
        "config": config,
        "rows": len(frame),
    }, indent=2), encoding="utf-8")


def run_backend_evaluation(backend: Backend) -> pd.DataFrame:
    results = load_checkpoint(backend)
    completed_ids = set(results["id"].astype(str))
    pending = evaluation_df.loc[~evaluation_df["id"].astype(str).isin(completed_ids)]
    print(f"{backend.name}: completed={len(completed_ids):,}, pending={len(pending):,}")
    buffered: list[dict[str, Any]] = []
    consecutive_failures = 0

    for position, (_, row) in enumerate(pending.iterrows(), start=1):
        started = time.perf_counter()
        try:
            prediction = backend.predict(email_state(row))
            elapsed_ms = (time.perf_counter() - started) * 1000
            max_probability = max(prediction.probabilities.values())
            result = {
                "id": row["id"],
                "expected_category": row["category"],
                "predicted_category": prediction.choice,
                "native_confidence": prediction.native_confidence,
                "max_probability": max_probability,
                "latency_ms": elapsed_ms,
                "reported_latency_ms": prediction.reported_latency_ms,
                "model": prediction.model,
                "input_tokens": prediction.input_tokens,
                "error": "",
                **{f"p_{label}": prediction.probabilities[label] for label in LABELS},
            }
            consecutive_failures = 0
        except (OSError, RuntimeError, ValueError, KeyError, TypeError, requests.RequestException) as exc:
            elapsed_ms = (time.perf_counter() - started) * 1000
            result = {
                "id": row["id"],
                "expected_category": row["category"],
                "predicted_category": pd.NA,
                "native_confidence": np.nan,
                "max_probability": np.nan,
                "latency_ms": elapsed_ms,
                "reported_latency_ms": np.nan,
                "model": backend.model_id,
                "input_tokens": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
                **{column: np.nan for column in probability_columns},
            }
            consecutive_failures += 1
        buffered.append(result)

        should_save = position % CHECKPOINT_EVERY == 0 or position == len(pending)
        if should_save or consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            results = pd.concat([results, pd.DataFrame(buffered)], ignore_index=True)
            results = results.drop_duplicates("id", keep="last")
            save_checkpoint(backend, results)
            buffered.clear()
            print(f"{backend.name}: saved {len(results):,}/{len(evaluation_df):,}")
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            raise RuntimeError(
                f"{backend.name} stopped after {consecutive_failures} consecutive failures. "
                f"Last error: {result['error']}"
            )
    return results.sort_values("id").reset_index(drop=True)


for backend_name, should_run in RUN_EVALUATIONS.items():
    if not should_run:
        continue
    if backend_name not in backends:
        raise RuntimeError(f"{backend_name} is marked to run but was not initialized successfully.")
    run_backend_evaluation(backends[backend_name])

## Load available checkpoints and compute comparable metrics

Completed checkpoints can be analyzed even when a backend is currently disabled. Quality is always reported alongside coverage.

In [ ]:
def load_available_results() -> dict[str, pd.DataFrame]:
    loaded: dict[str, pd.DataFrame] = {}
    for backend_name in BACKEND_BUILDERS:
        prediction_path, _ = checkpoint_paths(backend_name)
        if prediction_path.exists():
            frame = pd.read_csv(prediction_path)
            required = set(result_columns)
            missing = required.difference(frame.columns)
            if missing:
                raise ValueError(f"{prediction_path.name} is missing columns: {sorted(missing)}")
            loaded[backend_name] = frame.loc[frame["id"].astype(str).isin(evaluation_df["id"].astype(str))].copy()
    return loaded


def multiclass_brier(y_true: pd.Series, probabilities: np.ndarray) -> float:
    targets = np.zeros_like(probabilities)
    label_to_index = {label: index for index, label in enumerate(LABELS)}
    for row_index, label in enumerate(y_true):
        targets[row_index, label_to_index[label]] = 1.0
    return float(np.mean(np.sum((probabilities - targets) ** 2, axis=1)))


def expected_calibration_error(frame: pd.DataFrame, bins: int = 10) -> float:
    confidence = frame["max_probability"].to_numpy(dtype=float)
    correct = frame["predicted_category"].eq(frame["expected_category"]).to_numpy(dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for index in range(bins):
        if index == bins - 1:
            mask = (confidence >= edges[index]) & (confidence <= edges[index + 1])
        else:
            mask = (confidence >= edges[index]) & (confidence < edges[index + 1])
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


def summarize_backend(backend_name: str, frame: pd.DataFrame) -> tuple[dict[str, Any], pd.DataFrame]:
    successful = frame.loc[frame["error"].fillna("").eq("")].copy()
    coverage = len(successful) / len(evaluation_df)
    if successful.empty:
        return {"backend": backend_name, "rows": 0, "coverage": coverage}, successful
    y_true = successful["expected_category"]
    y_pred = successful["predicted_category"]
    y_prob = successful[probability_columns].to_numpy(dtype=float)
    summary = {
        "backend": backend_name,
        "rows": len(successful),
        "coverage": coverage,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0),
        "log_loss": log_loss(y_true, y_prob, labels=LABELS),
        "brier_score": multiclass_brier(y_true, y_prob),
        "top_2_accuracy": top_k_accuracy_score(y_true, y_prob, k=2, labels=LABELS),
        "mean_max_probability": successful["max_probability"].mean(),
        "ece_10_bin": expected_calibration_error(successful),
        "median_latency_ms": successful["latency_ms"].median(),
        "p95_latency_ms": successful["latency_ms"].quantile(0.95),
        "failures": len(frame) - len(successful),
    }
    return summary, successful


results_by_backend = load_available_results()
summaries: list[dict[str, Any]] = []
successful_by_backend: dict[str, pd.DataFrame] = {}
for backend_name, frame in results_by_backend.items():
    summary, successful = summarize_backend(backend_name, frame)
    summaries.append(summary)
    successful_by_backend[backend_name] = successful

summary_df = pd.DataFrame(summaries).set_index("backend") if summaries else pd.DataFrame()
if summary_df.empty:
    print("No comparison checkpoints are available. Run at least one backend evaluation.")
else:
    display(summary_df.sort_values("macro_f1", ascending=False))

## Per-class quality and confusion matrices

In [ ]:
per_class_frames: list[pd.DataFrame] = []
for backend_name, successful in successful_by_backend.items():
    if successful.empty:
        continue
    report = classification_report(
        successful["expected_category"],
        successful["predicted_category"],
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )
    per_class = pd.DataFrame(report).T.loc[LABELS, ["precision", "recall", "f1-score", "support"]]
    per_class.insert(0, "backend", backend_name)
    per_class.insert(1, "label", per_class.index)
    per_class_frames.append(per_class.reset_index(drop=True))

if per_class_frames:
    per_class_df = pd.concat(per_class_frames, ignore_index=True)
    display(per_class_df.pivot(index="label", columns="backend", values="f1-score").reindex(LABELS))

    figure, axes = plt.subplots(1, len(successful_by_backend), figsize=(7 * len(successful_by_backend), 6), squeeze=False)
    for axis, (backend_name, successful) in zip(axes[0], successful_by_backend.items()):
        if successful.empty:
            axis.set_visible(False)
            continue
        ConfusionMatrixDisplay.from_predictions(
            successful["expected_category"],
            successful["predicted_category"],
            labels=LABELS,
            normalize="true",
            values_format=".2f",
            cmap="Blues",
            colorbar=False,
            ax=axis,
        )
        axis.set_title(backend_name)
        axis.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()

## Calibration and latency

Calibration curves use maximum class probability, the one confidence quantity available in the same form for all systems. Latency is client-observed and therefore includes local model execution or Azure network time as applicable.

In [ ]:
if successful_by_backend:
    figure, axes = plt.subplots(1, 2, figsize=(15, 6))
    for backend_name, successful in successful_by_backend.items():
        if successful.empty:
            continue
        calibration = successful.assign(
            correct=successful["predicted_category"].eq(successful["expected_category"]),
            confidence_bin=pd.cut(successful["max_probability"], bins=np.linspace(0, 1, 11), include_lowest=True),
        ).groupby("confidence_bin", observed=False).agg(
            examples=("correct", "size"),
            mean_confidence=("max_probability", "mean"),
            empirical_accuracy=("correct", "mean"),
        ).dropna(subset=["mean_confidence"])
        axes[0].plot(calibration["mean_confidence"], calibration["empirical_accuracy"], marker="o", label=backend_name)
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean max probability", ylabel="Empirical accuracy", title="Calibration")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    latency_data = [frame["latency_ms"].to_numpy(dtype=float) for frame in successful_by_backend.values() if not frame.empty]
    latency_labels = [name for name, frame in successful_by_backend.items() if not frame.empty]
    axes[1].boxplot(latency_data, tick_labels=latency_labels, showfliers=False)
    axes[1].set(ylabel="Client-observed latency (ms)", title="Warm-run latency distribution")
    axes[1].grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Paired head-to-head comparison

Only IDs successfully predicted by both backends enter a pairwise comparison. The table reports label agreement, each backend's paired accuracy, an exact McNemar p-value, and a stratified bootstrap interval for the accuracy difference (`backend_a - backend_b`).

In [ ]:
def exact_mcnemar_pvalue(a_only_correct: int, b_only_correct: int) -> float:
    discordant = a_only_correct + b_only_correct
    if discordant == 0:
        return 1.0
    tail = sum(math.comb(discordant, k) for k in range(0, min(a_only_correct, b_only_correct) + 1))
    return min(1.0, 2.0 * tail / (2 ** discordant))


def stratified_accuracy_difference_interval(
    paired: pd.DataFrame,
    replicates: int = BOOTSTRAP_REPLICATES,
) -> tuple[float, float]:
    rng = np.random.default_rng(RANDOM_STATE)
    grouped_indices = [group.index.to_numpy() for _, group in paired.groupby("expected_category")]
    differences = np.empty(replicates)
    for replicate in range(replicates):
        sample_indices = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True) for indices in grouped_indices
        ])
        sample = paired.loc[sample_indices]
        differences[replicate] = sample["a_correct"].mean() - sample["b_correct"].mean()
    lower, upper = np.quantile(differences, [0.025, 0.975])
    return float(lower), float(upper)


pairwise_rows: list[dict[str, Any]] = []
backend_names = sorted(successful_by_backend)
for a_index, backend_a in enumerate(backend_names):
    for backend_b in backend_names[a_index + 1:]:
        a = successful_by_backend[backend_a][["id", "expected_category", "predicted_category"]].rename(
            columns={"predicted_category": "prediction_a"}
        )
        b = successful_by_backend[backend_b][["id", "expected_category", "predicted_category"]].rename(
            columns={"predicted_category": "prediction_b"}
        )
        paired = a.merge(b, on=["id", "expected_category"], how="inner", validate="one_to_one")
        if paired.empty:
            continue
        paired["a_correct"] = paired["prediction_a"].eq(paired["expected_category"])
        paired["b_correct"] = paired["prediction_b"].eq(paired["expected_category"])
        a_only = int((paired["a_correct"] & ~paired["b_correct"]).sum())
        b_only = int((~paired["a_correct"] & paired["b_correct"]).sum())
        lower, upper = stratified_accuracy_difference_interval(paired)
        pairwise_rows.append({
            "backend_a": backend_a,
            "backend_b": backend_b,
            "paired_rows": len(paired),
            "label_agreement": paired["prediction_a"].eq(paired["prediction_b"]).mean(),
            "accuracy_a": paired["a_correct"].mean(),
            "accuracy_b": paired["b_correct"].mean(),
            "accuracy_difference_a_minus_b": paired["a_correct"].mean() - paired["b_correct"].mean(),
            "difference_95ci_low": lower,
            "difference_95ci_high": upper,
            "a_only_correct": a_only,
            "b_only_correct": b_only,
            "mcnemar_exact_pvalue": exact_mcnemar_pvalue(a_only, b_only),
        })

if pairwise_rows:
    pairwise_df = pd.DataFrame(pairwise_rows)
    display(pairwise_df)
else:
    print("At least two completed backend checkpoints are required for paired comparison.")

## Comparative error analysis

The first table shows prediction counts, making label collapse visible. The second shows truncated content only for cases where available backends disagree.

In [ ]:
if successful_by_backend:
    prediction_counts = pd.DataFrame({
        backend_name: frame["predicted_category"].value_counts().reindex(LABELS, fill_value=0)
        for backend_name, frame in successful_by_backend.items()
    })
    display(prediction_counts)

if len(successful_by_backend) >= 2:
    comparison = evaluation_df[["id", "category", "subject", "body"]].rename(columns={"category": "expected_category"})
    prediction_columns = []
    for backend_name, frame in successful_by_backend.items():
        column = f"prediction_{backend_name}"
        prediction_columns.append(column)
        comparison = comparison.merge(
            frame[["id", "predicted_category"]].rename(columns={"predicted_category": column}),
            on="id",
            how="inner",
            validate="one_to_one",
        )
    disagreement = comparison.loc[comparison[prediction_columns].nunique(axis=1) > 1].copy()
    disagreement["subject_preview"] = disagreement["subject"].astype(str).str.slice(0, 120)
    disagreement["body_preview"] = disagreement["body"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 240)
    display(disagreement[["id", "expected_category", *prediction_columns, "subject_preview", "body_preview"]].head(50))
    print(f"Disagreements: {len(disagreement):,}/{len(comparison):,}")

## Reproducibility metadata and indicative Microsoft cost

Microsoft's published launch price was $0.042 per million input tokens with free output tokens. Confirm current pricing before using this estimate. Laya and CLM are local here, so their meaningful cost is measured infrastructure and operations rather than a universal token price.

In [ ]:
MICROSOFT_INPUT_USD_PER_MILLION = 0.042
cost_rows: list[dict[str, Any]] = []
for backend_name, successful in successful_by_backend.items():
    reported_tokens = pd.to_numeric(successful["input_tokens"], errors="coerce").dropna()
    input_tokens = int(reported_tokens.sum()) if not reported_tokens.empty else None
    indicative_cost = None
    if backend_name == "microsoft_decision_1" and input_tokens is not None:
        indicative_cost = input_tokens * MICROSOFT_INPUT_USD_PER_MILLION / 1_000_000
    cost_rows.append({
        "backend": backend_name,
        "reported_input_tokens": input_tokens,
        "indicative_microsoft_usd": indicative_cost,
        "note": "Hosted service estimate" if backend_name == "microsoft_decision_1" else "Measure local infrastructure cost",
    })
if cost_rows:
    display(pd.DataFrame(cost_rows))

run_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": DATASET_ID,
    "dataset_file": TEST_FILE,
    "test_rows": len(test_df),
    "evaluation_rows": len(evaluation_df),
    "random_state": RANDOM_STATE,
    "labels": LABELS,
    "criteria": CATEGORY_CRITERIA,
    "instruction": CATEGORY_INSTRUCTION,
    "enabled_backends": ENABLE_BACKENDS,
    "run_evaluations": RUN_EVALUATIONS,
    "clm_model": CLM_MODEL,
    "azure_deployment": AZURE_DECISION_DEPLOYMENT or None,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}
metadata_path = ARTIFACT_DIR / "email_three_way_run_metadata.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print(f"Saved metadata: {metadata_path}")

## Interpretation checklist

Before selecting a backend, document:

1. **Coverage and failures:** never compare quality without successful-request coverage.
2. **Primary quality:** macro F1 is important because every class must work; accuracy alone can hide label collapse.
3. **Probability quality:** compare log loss, Brier score, ECE, and calibration curves—not vendor claims.
4. **Class behavior:** identify weak labels and dominant confusion pairs.
5. **Paired evidence:** use the same IDs and inspect the accuracy-difference interval and McNemar result.
6. **Latency:** report median and p95 from repeated warm runs in the intended deployment topology.
7. **Cost:** combine service charges or measured GPU duty cycle with engineering and operational costs.
8. **Generalization:** this dataset is synthetic and has known train/test overlap. Repeat the comparison on real, privacy-reviewed production-like email.
9. **Tuning fairness:** compare zero-shot models first. If one system is fine-tuned, clearly label it and give equivalent training data and tuning opportunity to the alternatives where supported.
10. **Governance:** keep deterministic policy and human review outside all three models for consequential actions.